# Postać kanoniczna Jordana — przykład

Rozpatrujemy macierz z [przykładu w materiałach teoretycznych](../teoria/postac_jordana_przyklad.pdf). Wyznaczymy jej postać Jordana i macierz podobieństwa. Obliczenia symboliczne wykonamy w SymPy, a na końcu niezależnie sprawdzimy wynik numerycznie w NumPy.

In [1]:
import numpy as np
import sympy as sp
from IPython.display import display

A = sp.Matrix([
    [2, 0, 0, 0],
    [0, 1, 1, 0],
    [0, 0, 1, 0],
    [0, 2, -1, 2],
])

print("Dana macierz A:")
display(A)

Dana macierz A:


Matrix([
[2, 0,  0, 0],
[0, 1,  1, 0],
[0, 0,  1, 0],
[0, 2, -1, 2]])

## Wartości własne i rozmiary klatek

Wielomian charakterystyczny określa krotności algebraiczne wartości własnych. Wymiar jądra $A-\lambda I$ to krotność geometryczna, czyli liczba klatek Jordana dla wartości własnej $\lambda$.

In [2]:
lam = sp.symbols("lambda")
charpoly = A.charpoly(lam).as_expr()
print("Wielomian charakterystyczny:")
display(sp.factor(charpoly))

for eigenvalue in (1, 2):
    eigenspace = (A - eigenvalue * sp.eye(4)).nullspace()
    print(f"\nWartość własna {eigenvalue}:")
    print(f"Wymiar przestrzeni własnej: {len(eigenspace)}")
    display(sp.Matrix.hstack(*eigenspace))

Wielomian charakterystyczny:


(lambda - 2)**2*(lambda - 1)**2


Wartość własna 1:
Wymiar przestrzeni własnej: 1


Matrix([
[   0],
[-1/2],
[   0],
[   1]])


Wartość własna 2:
Wymiar przestrzeni własnej: 2


Matrix([
[1, 0],
[0, 0],
[0, 0],
[0, 1]])

Z rozkładu $\chi_A(\lambda)=(\lambda-1)^2(\lambda-2)^2$ wynika, że obie wartości własne mają krotność algebraiczną 2. Dla $\lambda=1$ przestrzeń własna ma wymiar 1, więc odpowiada jej jedna klatka rozmiaru 2. Dla $\lambda=2$ przestrzeń własna ma wymiar 2, więc dostajemy dwie klatki rozmiaru 1.

## Łańcuch wektorów dla $\lambda=1$

Dla klatki $\begin{bmatrix}1&1\\0&1\end{bmatrix}$ wybieramy wektor własny $v_1$ i wektor uogólniony $v_2$, tak aby $(A-I)v_1=0$ oraz $(A-I)v_2=v_1$. Dla $\lambda=2$ potrzebujemy dwóch liniowo niezależnych wektorów własnych.

In [3]:
v1 = sp.Matrix([0, 1, 0, -2])
v2 = sp.Matrix([0, 0, 1, -1])
w1 = sp.Matrix([1, 0, 0, 0])
w2 = sp.Matrix([0, 0, 0, 1])
I = sp.eye(4)

print("(A - I)v1 =")
display((A - I) * v1)
print("(A - I)v2 = v1:")
display((A - I) * v2)
print("Wektory własne dla lambda=2, sprawdzenie (A - 2I)w = 0:")
display(sp.Matrix.hstack((A - 2 * I) * w1, (A - 2 * I) * w2))

assert (A - I) * v1 == sp.zeros(4, 1)
assert (A - I) * v2 == v1
assert (A - 2 * I) * w1 == sp.zeros(4, 1)
assert (A - 2 * I) * w2 == sp.zeros(4, 1)

(A - I)v1 =


Matrix([
[0],
[0],
[0],
[0]])

(A - I)v2 = v1:


Matrix([
[ 0],
[ 1],
[ 0],
[-2]])

Wektory własne dla lambda=2, sprawdzenie (A - 2I)w = 0:


Matrix([
[0, 0],
[0, 0],
[0, 0],
[0, 0]])

## Macierz Jordana i macierz modalna

Kolumny macierzy modalnej $T$ ustawiamy w kolejności łańcuchów Jordana: najpierw $v_1,v_2$, potem wektory własne dla $\lambda=2$. Wówczas zachodzi $AT=TJ$, czyli $J=T^{-1}AT$.

In [4]:
J = sp.diag(
    sp.Matrix([[1, 1], [0, 1]]),
    sp.Matrix([[2]]),
    sp.Matrix([[2]]),
)
T = sp.Matrix.hstack(v1, v2, w1, w2)

print("Postać Jordana J:")
display(J)
print("Macierz modalna T:")
display(T)
print(f"det(T) = {T.det()}, więc T jest odwracalna.")
print("Kontrola AT - TJ:")
display(A * T - T * J)
print("Odzyskana macierz T^(-1) A T:")
display(T.inv() * A * T)

assert T.det() != 0
assert A * T == T * J
assert T.inv() * A * T == J

Postać Jordana J:


Matrix([
[1, 1, 0, 0],
[0, 1, 0, 0],
[0, 0, 2, 0],
[0, 0, 0, 2]])

Macierz modalna T:


Matrix([
[ 0,  0, 1, 0],
[ 1,  0, 0, 0],
[ 0,  1, 0, 0],
[-2, -1, 0, 1]])

det(T) = 1, więc T jest odwracalna.
Kontrola AT - TJ:


Matrix([
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0]])

Odzyskana macierz T^(-1) A T:


Matrix([
[1, 1, 0, 0],
[0, 1, 0, 0],
[0, 0, 2, 0],
[0, 0, 0, 2]])

## Weryfikacja numeryczna w NumPy

Przeliczamy macierze na typ zmiennoprzecinkowy i sprawdzamy tę samą relację z tolerancją numeryczną. W przykładzie wszystkie dane są całkowite, ale NumPy wykonuje obliczenia na liczbach zmiennoprzecinkowych.

In [5]:
A_np = np.array(A.tolist(), dtype=float)
J_np = np.array(J.tolist(), dtype=float)
T_np = np.array(T.tolist(), dtype=float)

similarity_residual = np.linalg.inv(T_np) @ A_np @ T_np - J_np
print("T^(-1) A T obliczone numerycznie:")
print(np.linalg.inv(T_np) @ A_np @ T_np)
print("Norma reszty ||T^(-1) A T - J||:")
print(np.linalg.norm(similarity_residual))

np.testing.assert_allclose(similarity_residual, np.zeros((4, 4)), atol=1e-12)

T^(-1) A T obliczone numerycznie:
[[1. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 2. 0.]
 [0. 0. 0. 2.]]
Norma reszty ||T^(-1) A T - J||:
0.0


## Wyznaczenie postaci Jordana w NumPy

NumPy nie ma gotowej funkcji do obliczania postaci Jordana. Dla tej macierzy wyznaczamy numerycznie wartości własne przez `eigvals`, przestrzenie własne jako jądra znalezione przez SVD, a następnie uzupełniamy wektor własny dla $\lambda=1$ do łańcucha Jordana, rozwiązując $(A-I)v_2=v_1$ metodą najmniejszych kwadratów. Z otrzymanych kolumn macierzy modalnej $T$ obliczamy $J=T^{-1}AT$. Ze względu na arytmetykę zmiennoprzecinkową wynik może zawierać małe błędy zaokrągleń; progi SVD i porównania trzeba dobrać do skali danych.

In [6]:
A_num = np.array([
    [2, 0, 0, 0],
    [0, 1, 1, 0],
    [0, 0, 1, 0],
    [0, 2, -1, 2],
], dtype=float)

eigenvalues_num = np.linalg.eigvals(A_num)
lambda_1 = float(np.mean(eigenvalues_num[np.isclose(eigenvalues_num, 1, atol=1e-8, rtol=0)]))
lambda_2 = float(np.mean(eigenvalues_num[np.isclose(eigenvalues_num, 2, atol=1e-8, rtol=0)]))

def null_space_svd(matrix, rtol=1e-12):
    _, singular_values, vh = np.linalg.svd(matrix)
    rank = np.count_nonzero(singular_values > rtol * singular_values[0])
    return vh[rank:].T

eigenspace_1 = null_space_svd(A_num - lambda_1 * np.eye(4))
eigenspace_2 = null_space_svd(A_num - lambda_2 * np.eye(4))
assert eigenspace_1.shape[1] == 1
assert eigenspace_2.shape[1] == 2

v1_num = eigenspace_1[:, 0]
first_nonzero = np.flatnonzero(np.abs(v1_num) > 1e-10)[0]
if v1_num[first_nonzero] < 0:
    v1_num = -v1_num
v2_num, *_ = np.linalg.lstsq(
    A_num - lambda_1 * np.eye(4), v1_num, rcond=None
)

T_num = np.column_stack((v1_num, v2_num, eigenspace_2))
J_num = np.linalg.solve(T_num, A_num @ T_num)
J_num_display = np.where(np.abs(J_num) < 1e-10, 0, J_num)

print("Wartości własne wyznaczone przez NumPy:", np.round(eigenvalues_num, 8))
print("Wymiary przestrzeni własnych:", eigenspace_1.shape[1], "i", eigenspace_2.shape[1])
print("Macierz Jordana wyznaczona numerycznie:")
print(np.round(J_num_display, 10))
print("Wyznacznik macierzy modalnej:", np.linalg.det(T_num))

J_expected = np.array([
    [1, 1, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 2, 0],
    [0, 0, 0, 2],
], dtype=float)
np.testing.assert_allclose(J_num, J_expected, atol=1e-10)
np.testing.assert_allclose((A_num - lambda_1 * np.eye(4)) @ v2_num, v1_num, atol=1e-10)

Wartości własne wyznaczone przez NumPy: [2. 2. 1. 1.]
Wymiary przestrzeni własnych: 1 i 2
Macierz Jordana wyznaczona numerycznie:
[[1. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 2. 0.]
 [0. 0. 0. 2.]]
Wyznacznik macierzy modalnej: -0.20000000000000007
